# 05 · Embedding: Moran's I, NMF, t-SNE, Harmony

Following the tutorial notebook [`05_embedding_student.ipynb`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/blob/main/notebooks/level2/05_embedding_student.ipynb).

**Kernel:** `cajal-lipidomics`

**What I'm doing here**
- Moran's I feature selection (> 0.4 in both sections) + permutation null
- Seeded NMF with 12 programs, learned on control, applied to both
- t-SNE for looking only, Harmony for clustering only (never for differential tests)
- Bonus MLP: predict pixel position from lipids. Save `05_embedded.h5ad`

**My notes / what surprised me**
- ...

In [ ]:
# ---- setup: the same first cell in every notebook ---------------------------------------

# My own little package. `paths` knows where the tutorial clone, its provided data,
# and my own outputs live, so I never type "../../data/..." by hand.
from lipid_project import paths

paths.ensure_dirs()   # creates data/derived/, figures/, results/ if they're missing (safe to re-run)
paths.summary()       # prints where everything points + which stages I've built so far.
                      # If a line starts with "--", that thing is missing: fix it before going on.

# The usual scientific stack
import numpy as np                 # arrays and math
import pandas as pd                # tables (obs / var are pandas DataFrames)
import matplotlib.pyplot as plt    # plotting
import anndata as ad               # the AnnData container: X (pixels x lipids) + obs + var + obsm + layers

# The tutorial's helper package (installed from my clone in external/).
# Rule from the course: before calling any cl.<module>.<function>, open
# external/lipidomics_tutorial_cajalcourse/src/cajal_lipidomics/<module>.py and read it.
import cajal_lipidomics as cl
cl.style.set_style()               # the course's figure defaults, so plots look clean and consistent

# If this prints without a red error, the kernel is right. A ModuleNotFoundError here almost
# always means the wrong kernel: re-pick "cajal-lipidomics" in the top-right kernel picker.
print("ready. cajal_lipidomics", cl.__version__)

## load

In [ ]:
# ---- load the previous stage --------------------------------------------------------------
# My NB03 output (NB04 didn't save a stage). Two versions of the data travel together:
#   adata.X                 raw ion intensities, kept for reference, NOT what I embed
#   adata.layers["umaia"]   uMAIA-normalized values, what I build the embedding from
adata = ad.read_h5ad(paths.stage("normalized"))

print("pixels x lipids:", adata.shape)
print("conditions:", dict(adata.obs["Condition"].value_counts()))
print("ions with a real lipid name:", int((~adata.var["lipid"].astype(str).str.startswith("ion_")).sum()))

# Plan for this notebook:
#   Moran's I feature selection (> 0.4 in BOTH sections)  ->  seeded NMF, 12 programs, learned
#   on control and applied to both  ->  t-SNE (just to look)  ->  Harmony (for clustering ONLY,
#   never for differential testing, because with 2 sections batch = condition).

# ---- output -----------------------------------------------------------------------------------
# Adds obsm["X_nmf"], obsm["X_tsne"], obsm["X_harmony"] + Moran scores in var:
#   adata.write_h5ad(paths.stage("embedded"))    ->  data/derived/05_embedded.h5ad

## my work

In [ ]:
# my code from here on, one small step per cell, and look at the output each time